# Get Hourly Weather Data

This notebook pulls hourly historical weather data for Chicago from the Open-Meteo archive API, creates date and hour join keys, and saves a cleaned weather dataset for merging with hourly bike demand data.

In [1]:
from pathlib import Path
import pandas as pd
import requests

BASE_DIR = Path("D:/Projects/chicago-bike-demand")
PROCESSED_DIR = BASE_DIR / "01_data" / "Processed"

print("Base directory:", BASE_DIR)
print("Processed directory:", PROCESSED_DIR)

Base directory: D:\Projects\chicago-bike-demand
Processed directory: D:\Projects\chicago-bike-demand\01_data\Processed


## Define weather data function

In [2]:
def get_weather_data(start_date: str, end_date: str) -> pd.DataFrame:
    url = "https://archive-api.open-meteo.com/v1/archive"

    params = {
        "latitude": 41.8781,   # Chicago
        "longitude": -87.6298,
        "start_date": start_date,
        "end_date": end_date,
        "hourly": [
            "temperature_2m",
            "precipitation",
            "windspeed_10m"
        ],
        "timezone": "America/Chicago"
    }

    response = requests.get(url, params=params, timeout=30)
    response.raise_for_status()

    data = response.json()

    if "hourly" not in data:
        raise ValueError("Expected hourly weather data was not returned by the API.")

    df_weather = pd.DataFrame(data["hourly"])

    return df_weather

## Pull weather data

In [3]:
start_date = "2025-02-28"
end_date = "2025-05-31"

df_weather = get_weather_data(start_date, end_date)

print("Weather data pulled successfully.")
print("Shape:", df_weather.shape)
df_weather.head()

Weather data pulled successfully.
Shape: (2232, 4)


,time,temperature_2m,precipitation,windspeed_10m
0,2025-02-28T00:00,1.8,0.0,8.9
1,2025-02-28T01:00,1.3,0.0,9.5
2,2025-02-28T02:00,1.0,0.0,10.7
3,2025-02-28T03:00,0.8,0.0,14.0
4,2025-02-28T04:00,0.9,0.0,14.1


## Convert timestamps and create join keys

In [4]:
df_weather["time"] = pd.to_datetime(df_weather["time"], errors="coerce")

df_weather["date"] = df_weather["time"].dt.date
df_weather["hour"] = df_weather["time"].dt.hour

df_weather[["time", "date", "hour"]].head()

,time,date,hour
0,2025-02-28 00:00:00,2025-02-28,0
1,2025-02-28 01:00:00,2025-02-28,1
2,2025-02-28 02:00:00,2025-02-28,2
3,2025-02-28 03:00:00,2025-02-28,3
4,2025-02-28 04:00:00,2025-02-28,4


## Rename weather columns

In [5]:
df_weather = df_weather.rename(columns={
    "temperature_2m": "temperature",
    "precipitation": "precipitation_mm",
    "windspeed_10m": "wind_speed"
})

df_weather.head()

,time,temperature,precipitation_mm,wind_speed,date,hour
0,2025-02-28 00:00:00,1.8,0.0,8.9,2025-02-28,0
1,2025-02-28 01:00:00,1.3,0.0,9.5,2025-02-28,1
2,2025-02-28 02:00:00,1.0,0.0,10.7,2025-02-28,2
3,2025-02-28 03:00:00,0.8,0.0,14.0,2025-02-28,3
4,2025-02-28 04:00:00,0.9,0.0,14.1,2025-02-28,4


## Validate weather data

In [6]:
print("Columns:")
print(df_weather.columns.tolist())

print("\nMissing values by column:")
print(df_weather.isna().sum())

print("\nDuplicate date-hour rows:")
print(df_weather[["date", "hour"]].duplicated().sum())

print("\nHour range check:")
print(df_weather["hour"].between(0, 23).all())

print("\nDate range:")
print("Min date:", df_weather["date"].min())
print("Max date:", df_weather["date"].max())

Columns:
['time', 'temperature', 'precipitation_mm', 'wind_speed', 'date', 'hour']

Missing values by column:
time                0
temperature         0
precipitation_mm    0
wind_speed          0
date                0
hour                0
dtype: int64

Duplicate date-hour rows:
0

Hour range check:
True

Date range:
Min date: 2025-02-28
Max date: 2025-05-31


## Save cleaned weather data

In [7]:
output_path = PROCESSED_DIR / "weather_hourly.csv"
output_path.parent.mkdir(parents=True, exist_ok=True)

df_weather.to_csv(output_path, index=False)

print("Weather data saved")
print(output_path)

Weather data saved
D:\Projects\chicago-bike-demand\01_data\Processed\weather_hourly.csv


## Final output check

In [8]:
check_weather = pd.read_csv(output_path)

print("Saved file shape:", check_weather.shape)
check_weather.head()

Saved file shape: (2232, 6)


,time,temperature,precipitation_mm,wind_speed,date,hour
0,2025-02-28 00:00:00,1.8,0.0,8.9,2025-02-28,0
1,2025-02-28 01:00:00,1.3,0.0,9.5,2025-02-28,1
2,2025-02-28 02:00:00,1.0,0.0,10.7,2025-02-28,2
3,2025-02-28 03:00:00,0.8,0.0,14.0,2025-02-28,3
4,2025-02-28 04:00:00,0.9,0.0,14.1,2025-02-28,4


## Weather Data Summary

Hourly historical weather data for Chicago was pulled from the Open-Meteo archive API.  
Timestamp fields were converted, date and hour join keys were created, and weather variables were renamed for easier downstream analysis.

This dataset is ready to be joined with city-level hourly bike demand data.